# Week 69, Day 6: Project review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w069d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('tiny_lex.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\n\n\nclass TinyError(Exception):\n    """Base class for every error the Tiny interpreter reports."""\n\n    def __init__(self, message, line=None, column=None):\n        super().__init__(message)\n        self.message, self.line, self.column = message, line, column\n\n    def __str__(self):\n        if self.line is None:\n            return self.message\n        if self.column is None:\n            return f"line {self.line}: {self.message}"\n        return f"line {self.line}, column {self.column}: {self.message}"\n\n\nclass LexError(TinyError):\n    """The source contains characters that don\'t form a token."""\n\n\nclass ParseError(TinyError):\n    """The tokens don\'t follow Tiny\'s grammar."""\n\n\nclass TinyRuntimeError(TinyError):\n    """Something went wrong while a program ran."""\n\n\ndef show_error(source, error):\n    if error.line is None:\n        return str(error)\n    text = source.split("\\n")[error.line - 1]\n    if error.column is None:\n        return f"{error}\\n    {text}"\n    return f"{error}\\n    {text}\\n    {\' \' * (error.column - 1)}^"\n\n\n@dataclass(frozen=True)\nclass Token:\n    kind: str\n    text: str\n    value: object\n    line: int\n    column: int\n\n\nclass Cursor:\n    def __init__(self, source):\n        self.source, self.pos, self.line, self.column = source, 0, 1, 1\n\n    def at_end(self):\n        return self.pos >= len(self.source)\n\n    def peek(self, ahead=0):\n        i = self.pos + ahead\n        return self.source[i] if i < len(self.source) else ""\n\n    def advance(self):\n        ch = self.source[self.pos]\n        self.pos += 1\n        if ch == "\\n":\n            self.line, self.column = self.line + 1, 1\n        else:\n            self.column += 1\n        return ch\n\n\ndef read_number(cursor):\n    line, column, text = cursor.line, cursor.column, ""\n    while cursor.peek().isdigit():\n        text += cursor.advance()\n    if cursor.peek() == "." and cursor.peek(1).isdigit():\n        text += cursor.advance()\n        while cursor.peek().isdigit():\n            text += cursor.advance()\n        return Token("NUMBER", text, float(text), line, column)\n    return Token("NUMBER", text, int(text), line, column)\n\n\nESCAPES = {"n": "\\n", "t": "\\t", \'"\': \'"\', "\\\\": "\\\\"}\n\n\ndef read_string(cursor):\n    line, column = cursor.line, cursor.column\n    text = cursor.advance()\n    value = ""\n    while True:\n        if cursor.at_end() or cursor.peek() == "\\n":\n            raise LexError("unterminated string", line, column)\n        ch = cursor.advance()\n        text += ch\n        if ch == \'"\':\n            return Token("STRING", text, value, line, column)\n        if ch == "\\\\":\n            if cursor.at_end():\n                raise LexError("unterminated string", line, column)\n            esc_line, esc_col = cursor.line, cursor.column - 1\n            code = cursor.advance()\n            text += code\n            if code not in ESCAPES:\n                raise LexError(f"unknown escape \\\\{code}", esc_line, esc_col)\n            value += ESCAPES[code]\n        else:\n            value += ch\n\n\nKEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}\n\n\nTWO_CHAR = {"==", "!=", "<=", ">="}\n\n\nONE_CHAR = set("+-*/%<>=(){},;")\n\n\ndef read_name(cursor):\n    line, column, text = cursor.line, cursor.column, ""\n    while cursor.peek().isalnum() or cursor.peek() == "_":\n        text += cursor.advance()\n    if text in KEYWORDS:\n        value = {"true": True, "false": False, "nil": None}.get(text)\n        return Token(text.upper(), text, value, line, column)\n    return Token("NAME", text, text, line, column)\n\n\ndef read_operator(cursor):\n    line, column = cursor.line, cursor.column\n    pair = cursor.peek() + cursor.peek(1)\n    if pair in TWO_CHAR:\n        cursor.advance()\n        cursor.advance()\n        return Token("OP", pair, pair, line, column)\n    ch = cursor.peek()\n    if ch in ONE_CHAR:\n        cursor.advance()\n        return Token("OP", ch, ch, line, column)\n    raise LexError(f"unexpected character {ch!r}", line, column)\n\n\ndef skip_space_and_comments(cursor):\n    while not cursor.at_end():\n        ch = cursor.peek()\n        if ch in " \\t\\r\\n":\n            cursor.advance()\n        elif ch == "#":\n            while not cursor.at_end() and cursor.peek() != "\\n":\n                cursor.advance()\n        else:\n            return\n\n\ndef tokenize(source):\n    cursor = Cursor(source)\n    tokens = []\n    while True:\n        skip_space_and_comments(cursor)\n        if cursor.at_end():\n            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))\n            return tokens\n        ch = cursor.peek()\n        if ch.isdigit():\n            tokens.append(read_number(cursor))\n        elif ch == \'"\':\n            tokens.append(read_string(cursor))\n        elif ch.isalpha() or ch == "_":\n            tokens.append(read_name(cursor))\n        else:\n            tokens.append(read_operator(cursor))\n')
with open('tiny_ast.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\n\n\n@dataclass\nclass Number:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass String:\n    value: str\n    line: int = 0\n\n\n@dataclass\nclass Literal:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Name:\n    name: str\n    line: int = 0\n\n\n@dataclass\nclass Unary:\n    op: str\n    operand: object\n    line: int = 0\n\n\n@dataclass\nclass Binary:\n    op: str\n    left: object\n    right: object\n    line: int = 0\n\n\n@dataclass\nclass Logical:\n    op: str\n    left: object\n    right: object\n    line: int = 0\n\n\n@dataclass\nclass Call:\n    callee: object\n    args: list\n    line: int = 0\n\n\n@dataclass\nclass Let:\n    name: str\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Assign:\n    name: str\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Print:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass ExprStmt:\n    value: object\n    line: int = 0\n\n\n@dataclass\nclass Block:\n    body: list\n    line: int = 0\n\n\n@dataclass\nclass If:\n    test: object\n    then: object\n    otherwise: object\n    line: int = 0\n\n\n@dataclass\nclass While:\n    test: object\n    body: object\n    line: int = 0\n\n\n@dataclass\nclass Function:\n    name: str\n    params: list\n    body: list\n    line: int = 0\n\n\n@dataclass\nclass Return:\n    value: object\n    line: int = 0\n')
with open('tiny_parse.py', 'w', encoding='utf-8') as f:
    f.write('from tiny_lex import *\nfrom tiny_ast import *\n\n\nclass Parser:\n    def __init__(self, tokens):\n        self.tokens, self.pos = tokens, 0\n\n    def peek(self):\n        return self.tokens[self.pos]\n\n    def at(self, kind, text=None):\n        tok = self.peek()\n        return tok.kind == kind and (text is None or tok.text == text)\n\n    def match(self, kind, text=None):\n        if self.at(kind, text):\n            tok = self.peek()\n            self.pos += 1\n            return tok\n        return None\n\n    def expect(self, kind, text=None, what=None):\n        tok = self.match(kind, text)\n        if tok is None:\n            got = self.peek()\n            shown = "end of input" if got.kind == "EOF" else repr(got.text)\n            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)\n        return tok\n\n    # --- expressions, lowest precedence first ---\n    def expression(self):\n        return self.logic_or()\n\n    def logic_or(self):\n        node = self.logic_and()\n        while self.at("OR"):\n            line = self.match("OR").line\n            node = Logical("or", node, self.logic_and(), line)\n        return node\n\n    def logic_and(self):\n        node = self.equality()\n        while self.at("AND"):\n            line = self.match("AND").line\n            node = Logical("and", node, self.equality(), line)\n        return node\n\n    def binary(self, operators, operand):\n        node = operand()\n        while self.peek().kind == "OP" and self.peek().text in operators:\n            tok = self.match("OP")\n            node = Binary(tok.text, node, operand(), tok.line)\n        return node\n\n    def equality(self):\n        return self.binary({"==", "!="}, self.comparison)\n\n    def comparison(self):\n        return self.binary({"<", "<=", ">", ">="}, self.term)\n\n    def term(self):\n        return self.binary({"+", "-"}, self.factor)\n\n    def factor(self):\n        return self.binary({"*", "/", "%"}, self.unary)\n\n    def unary(self):\n        if self.at("OP", "-"):\n            line = self.match("OP").line\n            return Unary("-", self.unary(), line)\n        if self.at("NOT"):\n            line = self.match("NOT").line\n            return Unary("not", self.unary(), line)\n        return self.call()\n\n    def call(self):\n        node = self.primary()\n        while self.at("OP", "("):\n            line = self.match("OP").line\n            args = []\n            if not self.at("OP", ")"):\n                args.append(self.expression())\n                while self.match("OP", ","):\n                    args.append(self.expression())\n            self.expect("OP", ")")\n            node = Call(node, args, line)\n        return node\n\n    def primary(self):\n        tok = self.peek()\n        if self.match("NUMBER"):\n            return Number(tok.value, tok.line)\n        if self.match("STRING"):\n            return String(tok.value, tok.line)\n        if tok.kind in ("TRUE", "FALSE", "NIL"):\n            self.pos += 1\n            return Literal(tok.value, tok.line)\n        if self.match("NAME"):\n            return Name(tok.text, tok.line)\n        if self.match("OP", "("):\n            node = self.expression()\n            self.expect("OP", ")")\n            return node\n        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)\n        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)\n\n    # --- statements ---\n    def statement(self):\n        tok = self.peek()\n        if self.match("LET"):\n            name = self.expect("NAME", what="a name").text\n            self.expect("OP", "=")\n            value = self.expression()\n            self.expect("OP", ";")\n            return Let(name, value, tok.line)\n        if self.match("PRINT"):\n            value = self.expression()\n            self.expect("OP", ";")\n            return Print(value, tok.line)\n        if self.at("OP", "{"):\n            return Block(self.block(), tok.line)\n        if self.match("IF"):\n            self.expect("OP", "(")\n            test = self.expression()\n            self.expect("OP", ")")\n            then = self.statement()\n            otherwise = self.statement() if self.match("ELSE") else None\n            return If(test, then, otherwise, tok.line)\n        if self.match("WHILE"):\n            self.expect("OP", "(")\n            test = self.expression()\n            self.expect("OP", ")")\n            return While(test, self.statement(), tok.line)\n        if self.match("FN"):\n            name = self.expect("NAME", what="a function name").text\n            self.expect("OP", "(")\n            params = []\n            if not self.at("OP", ")"):\n                params.append(self.expect("NAME", what="a parameter name").text)\n                while self.match("OP", ","):\n                    params.append(self.expect("NAME", what="a parameter name").text)\n            self.expect("OP", ")")\n            return Function(name, params, self.block(), tok.line)\n        if self.match("RETURN"):\n            value = None if self.at("OP", ";") else self.expression()\n            self.expect("OP", ";")\n            return Return(value, tok.line)\n        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":\n            self.pos += 2\n            value = self.expression()\n            self.expect("OP", ";")\n            return Assign(tok.text, value, tok.line)\n        value = self.expression()\n        self.expect("OP", ";")\n        return ExprStmt(value, tok.line)\n\n    def block(self):\n        self.expect("OP", "{")\n        body = []\n        while not self.at("OP", "}"):\n            if self.at("EOF"):\n                tok = self.peek()\n                raise ParseError("expected \'}\', got end of input", tok.line, tok.column)\n            body.append(self.statement())\n        self.expect("OP", "}")\n        return body\n\n    def program(self):\n        body = []\n        while not self.at("EOF"):\n            body.append(self.statement())\n        return body\n\n\ndef parse(source):\n    return Parser(tokenize(source)).program()\n')
with open('tiny_eval.py', 'w', encoding='utf-8') as f:
    f.write('from tiny_lex import *\nfrom tiny_ast import *\nfrom tiny_parse import parse\n\n\nclass Environment:\n    def __init__(self, parent=None):\n        self.values, self.parent = {}, parent\n\n    def define(self, name, value):\n        self.values[name] = value\n\n    def find(self, name):\n        env = self\n        while env is not None:\n            if name in env.values:\n                return env\n            env = env.parent\n        return None\n\n    def get(self, name, line=None):\n        env = self.find(name)\n        if env is None:\n            raise TinyRuntimeError(f"undefined name {name!r}", line)\n        return env.values[name]\n\n    def assign(self, name, value, line=None):\n        env = self.find(name)\n        if env is None:\n            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)\n        env.values[name] = value\n\n\nclass ReturnSignal(Exception):\n    def __init__(self, value):\n        self.value = value\n\n\nclass TinyFunction:\n    def __init__(self, decl, closure):\n        self.decl, self.closure = decl, closure\n\n    def __repr__(self):\n        return f"<fn {self.decl.name}>"\n\n\nclass Builtin:\n    def __init__(self, name, arity, func):\n        self.name, self.arity, self.func = name, arity, func\n\n    def __repr__(self):\n        return f"<builtin {self.name}>"\n\n\ndef truthy(value):\n    return value is not None and value is not False\n\n\ndef show(value):\n    if value is None:\n        return "nil"\n    if value is True:\n        return "true"\n    if value is False:\n        return "false"\n    if isinstance(value, float) and value.is_integer():\n        return str(int(value)) if abs(value) < 1e16 else str(value)\n    return str(value)\n\n\ndef type_name(value):\n    if value is None:\n        return "nil"\n    if isinstance(value, bool):\n        return "bool"\n    if isinstance(value, (int, float)):\n        return "number"\n    if isinstance(value, str):\n        return "string"\n    return "function"\n\n\ndef is_number(value):\n    return isinstance(value, (int, float)) and not isinstance(value, bool)\n\n\nclass Interpreter:\n    def __init__(self, output=None, max_depth=100):\n        self.output = [] if output is None else output\n        self.globals = Environment()\n        self.max_depth, self.depth = max_depth, 0\n        self.steps = 0\n        for name, arity, func in [\n            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),\n        ]:\n            self.globals.define(name, Builtin(name, arity, func))\n\n    @staticmethod\n    def _len(value):\n        if not isinstance(value, str):\n            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")\n        return len(value)\n\n    # --- expressions ---\n    def evaluate(self, node, env):\n        self.steps += 1\n        if isinstance(node, (Number, String, Literal)):\n            return node.value\n        if isinstance(node, Name):\n            return env.get(node.name, node.line)\n        if isinstance(node, Unary):\n            value = self.evaluate(node.operand, env)\n            if node.op == "-":\n                if not is_number(value):\n                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)\n                return -value\n            return not truthy(value)\n        if isinstance(node, Logical):\n            left = self.evaluate(node.left, env)\n            if node.op == "or":\n                return left if truthy(left) else self.evaluate(node.right, env)\n            return left if not truthy(left) else self.evaluate(node.right, env)\n        if isinstance(node, Binary):\n            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))\n        if isinstance(node, Call):\n            return self.call(node, env)\n        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))\n\n    def binary(self, node, left, right):\n        op = node.op\n        if op == "==":\n            return type_name(left) == type_name(right) and left == right\n        if op == "!=":\n            return not (type_name(left) == type_name(right) and left == right)\n        if op == "+" and isinstance(left, str) and isinstance(right, str):\n            return left + right\n        if not (is_number(left) and is_number(right)):\n            raise TinyRuntimeError(\n                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)\n        if op in ("/", "%") and right == 0:\n            raise TinyRuntimeError("division by zero", node.line)\n        if op == "/":\n            result = left / right\n            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result\n        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,\n                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,\n                ">": lambda: left > right, ">=": lambda: left >= right}[op]()\n\n    def call(self, node, env):\n        callee = self.evaluate(node.callee, env)\n        args = [self.evaluate(a, env) for a in node.args]\n        if isinstance(callee, Builtin):\n            if len(args) != callee.arity:\n                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)\n            try:\n                return callee.func(*args)\n            except TinyRuntimeError as e:\n                raise TinyRuntimeError(e.message, node.line) from None\n        if not isinstance(callee, TinyFunction):\n            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)\n        decl = callee.decl\n        if len(args) != len(decl.params):\n            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)\n        if self.depth >= self.max_depth:\n            raise TinyRuntimeError("too much recursion", node.line)\n        local = Environment(callee.closure)\n        for name, value in zip(decl.params, args):\n            local.define(name, value)\n        self.depth += 1\n        try:\n            self.run_block(decl.body, local)\n        except ReturnSignal as signal:\n            return signal.value\n        finally:\n            self.depth -= 1\n        return None\n\n    # --- statements ---\n    def execute(self, node, env):\n        self.steps += 1\n        if isinstance(node, Let):\n            env.define(node.name, self.evaluate(node.value, env))\n        elif isinstance(node, Assign):\n            env.assign(node.name, self.evaluate(node.value, env), node.line)\n        elif isinstance(node, Print):\n            self.output.append(show(self.evaluate(node.value, env)))\n        elif isinstance(node, ExprStmt):\n            self.evaluate(node.value, env)\n        elif isinstance(node, Block):\n            self.run_block(node.body, Environment(env))\n        elif isinstance(node, If):\n            if truthy(self.evaluate(node.test, env)):\n                self.execute(node.then, env)\n            elif node.otherwise is not None:\n                self.execute(node.otherwise, env)\n        elif isinstance(node, While):\n            while truthy(self.evaluate(node.test, env)):\n                self.execute(node.body, env)\n        elif isinstance(node, Function):\n            env.define(node.name, TinyFunction(node, env))\n        elif isinstance(node, Return):\n            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))\n        else:\n            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))\n\n    def run_block(self, statements, env):\n        for stmt in statements:\n            self.execute(stmt, env)\n\n    def run(self, program):\n        try:\n            self.run_block(program, self.globals)\n        except ReturnSignal:\n            raise TinyRuntimeError("return outside a function") from None\n        return self.output\n\n\ndef run_source(source):\n    interpreter = Interpreter()\n    interpreter.run(parse(source))\n    return interpreter.output\n')

### Bug hunt: Review: assignment that creates

Setting an existing variable from inside a block should change it where it was defined, but this `assign` puts a **new** variable in the current scope, so the outer `total` never changes and this prints `0`. Make `assign` change the variable in the scope that has it (and still fail for undefined names) so this prints `6`.

In [ ]:
class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent
    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None
    def assign(self, name, value):
        if self.find(name) is None:
            raise NameError(name)
        self.values[name] = value

outer = Environment()
outer.values["total"] = 0
for amount in (1, 2, 3):
    block = Environment(outer)
    block.assign("total", block.find("total").values["total"] + amount)
print(outer.values["total"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"assign where defined\"\n    expect_output(\"6\")")

### Bug hunt: Review: subtraction grouped the wrong way

This version of `term` handles the rest of the expression by calling itself, which groups to the **right**: `10 - 4 - 3` becomes `10 - (4 - 3)` and this prints `9`. Use a loop that builds to the left, as the grammar's `*` means, so it prints `3`.

In [ ]:
def term(tokens):
    left = int(tokens.pop(0))
    if tokens and tokens[0] in "+-":
        op = tokens.pop(0)
        right = term(tokens)
        return left - right if op == "-" else left + right
    return left

print(term("10 - 4 - 3".split()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"left grouping\"\n    expect_output(\"3\")")

### Bug hunt: Review: an or that isn't lazy

This evaluator works out **both** sides of `or` before deciding, so `found or lookup()` calls `lookup` even when `found` is already true, and the program prints `['lookup called']`. Evaluate the right side only when the left one is false, so it prints `[]`.

In [ ]:
calls = []

def lookup():
    calls.append("lookup called")
    return "from the table"

def evaluate_or(left_fn, right_fn):
    left, right = left_fn(), right_fn()
    return left if left not in (None, False) else right

result = evaluate_or(lambda: "cached", lookup)
print(calls)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"short-circuit\"\n    expect_output(\"[]\")")

### Bug hunt: Review: a closure that looks in the wrong place

Calling a function should run its body in a scope whose parent is the environment where the function was **defined**, but this `call` uses the caller's environment instead (so-called dynamic scope). `shout` was defined where `word` is `"hi"`, so it should print `HI`, but because it's called from a scope where `word` is `"bye"`, this prints `BYE`. Use the closure's environment so it prints `HI`.

In [ ]:
class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent
    def get(self, name):
        env = self
        while name not in env.values:
            env = env.parent
        return env.values[name]

class Function:
    def __init__(self, body, closure):
        self.body, self.closure = body, closure

def call(function, caller_env):
    local = Environment(caller_env)
    return function.body(local)

definer = Environment()
definer.values["word"] = "hi"
shout = Function(lambda env: env.get("word").upper(), definer)
caller = Environment()
caller.values["word"] = "bye"
print(call(shout, caller))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"lexical scope\"\n    expect_output(\"HI\")")

### Exercise: Add assert

Add an `assert` statement to Tiny **without editing** the existing classes, by extending them: `assert test;` or `assert test, message;` raises `TinyRuntimeError("assertion failed")` (or `"assertion failed: "` followed by the message shown with `show`) with the statement's line when the test isn't truthy, and does nothing otherwise. You'll need: `"assert"` added to `KEYWORDS` (so the tokeniser makes `ASSERT` tokens); an `Assert` dataclass (`test`, `message`, `line`); `AssertParser(Parser)` whose `statement` handles `assert` and otherwise defers to the parent (blocks call `self.statement`, so asserts inside functions work too); `AssertInterpreter(Interpreter)` whose `execute` handles `Assert` (counting a step) and otherwise defers; and `run_checked(source)`. The whole interpreter is given.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"


def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"


# your additions below
___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_assert():\n    \"a new statement\"\n    assert run_checked(\"let x = 2; assert x == 2; print x;\") == [\"2\"]\n    for src, message, line in ((\"assert 1 > 2;\", \"assertion failed\", 1),\n                               ('let n = nil;\\nassert n, \"n is \" + str(n);', \"assertion failed: n is nil\", 2),\n                               ('fn check(v) { assert v > 0, \"needs a positive number\"; return v; }\\nprint check(5);\\ncheck(-1);',\n                                \"assertion failed: needs a positive number\", 1)):\n        try:\n            run_checked(src)\n            assert False, src\n        except TinyRuntimeError as e:\n            assert e.line == line\n            if message:\n                assert e.message == message, e.message\n    try:\n        run_checked(\"assert;\")\n        assert False\n    except ParseError:\n        pass\n    assert run_checked('assert 0, \"zero is true in Tiny\";') == []\n    assert run_source(\"print 1;\") == [\"1\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## What you built

About 600 lines of Python that run a real (small) programming language, in four parts:

## A review checklist

Go through this list for your own interpreter, honestly.

**Does it meet the brief?** Monday listed every value, expression and statement. Try each one, especially the odd corners: `0` and `""` being true, `1 == true` being false, `else` attaching to the nearest `if`, `6 / 3` printing `2`.

**Are errors helpful?** Does every error, in every stage, say what was wrong and where? Is there any input that crashes the interpreter with a Python traceback instead of a `TinyError`? (Try very deep recursion, an empty file, a file that's just `}`.)

**Are the stages separate?** Does the tokeniser know anything about grammar, or the parser anything about values? Could you replace the evaluator (with a compiler to Week 64's stack machine, say) without touching the front end?

**Is scope right?** Do blocks hide their variables? Does assignment change the right variable? Do closures see their defining scope, and does each call get its own variables?

**Is it tested?** Would the tests catch each of the four review bugs below? Do golden tests cover every statement?

## The decisions that paid off

- **Errors first, with positions.** Every stage had an obvious way to fail, and every failure points at the source.
- **A pipeline of stages.** Each one was finished and tested before the next began, and each can be understood alone.
- **Data in dataclasses.** Tokens and tree nodes are plain records, easy to build in tests, compare and print.
- **One method per grammar rule.** The parser reads like the grammar, and precedence is simply the order of the methods.
- **Environments as linked scopes.** Blocks, functions and closures all fell out of one small class.
- **Tests before optimisation.** Constant folding changed the program being run, and the golden tests proved nothing visible changed.

## Reviewing code

Each of these is a small, self-contained version of a piece of the interpreter, with a mistake that's easy to make. Find it and fix it.

## One more feature: assert

## Where it could go next

- **Lists and indexing** (`[1, 2, 3]`, `xs[0]`): a new literal, a new kind of expression after `primary`, and a new value type.
- **A resolver pass** that works out, before running, which scope each name belongs to, so lookups don't have to walk the chain (the same idea as CPython's `LOAD_FAST`).
- **Compiling to bytecode** for Week 64's stack machine instead of walking the tree: usually several times faster.
- **Better errors**: "did you mean `total`?" suggestions for undefined names (Week 44's edit distance), and several errors per run from the parser too.

Robert Nystrom's free online book *Crafting Interpreters* builds a language much like Tiny in great depth, first as a tree-walker and then as a bytecode virtual machine; it's an excellent next step.

## Project quiz

## Week 69 review

Tick off each one honestly. If one feels shaky, go back and redo that day's steps.

- I can plan an interpreter as stages, and give every error a class and a position. ([Day 1](w069d1.html))
- I can write a tokeniser with a cursor, maximal munch, keywords, comments and error recovery. ([Day 2](w069d2.html))
- I can write a recursive-descent parser with precedence, statements, control flow and functions. ([Day 3](w069d3.html))
- I can write an evaluator with environments, closures, return and built-ins. ([Day 4](w069d4.html))
- I can test an interpreter against broken versions and golden outputs, profile it, and optimise it safely. ([Day 5](w069d5.html))
- I reviewed the interpreter, fixed four classic mistakes, and added assert by extension. (Today)

Tomorrow is a **rest day**. Next week: the **Advanced exam**: three days of review (iteration; CPython; concurrency), a practice exam, the exam itself, and a debrief.